# Proyecto Big Data - Steam Reviews (Dask)


In [1]:
# ==========================================================
# CONFIGURACION DE RECOLECCION DE TIEMPOS
# ==========================================================
# Arquitectura B: 1 Master + 4 Workers (n2-standard-4)

tiempos_resultados = {}
arquitectura = "4_workers"


In [2]:
!gsutil cp gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv /tmp/steam_reviews_500k.csv

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv...
/ [1 files][667.7 MiB/667.7 MiB]                                                
Operation completed over 1 objects/667.7 MiB.                                    


In [3]:
!pip install -U "pandas<2.2.0" "dask[dataframe]" "dask[distributed]" fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.12.2
    Uninstalling fsspec-2023.12.2:
      Successfully uninstalled fsspec-2023.12.2
  Attempting uninstall: dask
    Found existing installation: dask 2023.12.1━━━━━━━━━━━━━━━━━━━ 1/3 [dask]
    Uninstalling dask-2023.12.1:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [dask]
      Successfully uninstalled dask-2023.12.1━━━━━━━━━━━━━━━━━ 1/3 [dask]
  Attempting uninstall: distributed━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [dask]
    Found existing installation: distributed 2023.12.1━━━━━━━━ 1/3 [dask]
    Uninstalling distributed-2023.12.1:╸━━━━━━━━━━━━━ 2/3 [distributed]
      Successfully uninstalled distributed-2023.12.1━━━━━━━━━━ 2/3 [distributed]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [distributed] [distributed]
ERROR: pip's dependency resolver does not currently take into

In [4]:
from dask.distributed import Client

client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 31.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42451,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34233,Total threads: 1
Dashboard: http://127.0.0.1:45341/status,Memory: 7.84 GiB
Nanny: tcp://127.0.0.1:34303,


2026-09-21 15:47:45,467 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f94a304801e1d4e1354db31b8c5c291f initialized by task ('shuffle-transfer-f94a304801e1d4e1354db31b8c5c291f', 6) executed on worker tcp://127.0.0.1:43041
2026-09-21 15:47:50,792 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f94a304801e1d4e1354db31b8c5c291f deactivated due to stimulus 'task-finished-1790005670.790841'
2026-09-21 15:47:57,972 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f94a304801e1d4e1354db31b8c5c291f initialized by task ('shuffle-transfer-f94a304801e1d4e1354db31b8c5c291f', 9) executed on worker tcp://127.0.0.1:46633
2026-09-21 15:48:03,272 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f94a304801e1d4e1354db31b8c5c291f deactivated due to stimulus 'task-finished-1790005683.2703118'
2026-09-21 15:48:10,333 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f94a304801e1d4e1354db31b8c5c291f initialized by task ('shuffle-transfer-f94a304801e1d4e

In [5]:
import dask.dataframe as dd

dfd = dd.read_csv(
    "/tmp/steam_reviews_500k.csv",
    dtype="object"
)
print(dfd.npartitions)
print(dfd.head())

10
  recommendationid    appid               game     author_steamid  \
0        136840161  1551360    Forza Horizon 5  76561198053053492   
1        122064428  1808060         Obama Maze  76561199161385701   
2        138984652  1644320   Railway Empire 2  76561198031407785   
3         41914145   500710  Wild Terra Online  76561198031818080   
4        125403806   294100           RimWorld  76561198236110432   

  author_num_games_owned author_num_reviews author_playtime_forever  \
0                      0                 13                    8617   
1                     15                  7                      37   
2                    171                 55                    5843   
3                    215                  5                      19   
4                      0                 11                      86   

  author_playtime_last_two_weeks author_playtime_at_review author_last_played  \
0                              0                      8583         1684119

## Consulta 1 - Exploración y validación del dataset

Objetivo: conocer dimensiones, estructura, tipos de datos y valores nulos del dataset.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [6]:
# ==========================================================
# CONSULTA 1 - DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")


print("\nDimensiones del dataset")


filas = dfd.shape[0].compute()

print("Filas:", filas)

print("Columnas:", dfd.shape[1])


print("\nEstructura de datos")

print(dfd.dtypes)


print("\nValores nulos")


nulos = (
    dfd.isnull()
       .sum()
       .compute()
)

print(nulos)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_1"] = fin - inicio


===== DASK =====

Dimensiones del dataset
Filas: 500000
Columnas: 24

Estructura de datos
recommendationid                  string[pyarrow]
appid                             string[pyarrow]
game                              string[pyarrow]
author_steamid                    string[pyarrow]
author_num_games_owned            string[pyarrow]
author_num_reviews                string[pyarrow]
author_playtime_forever           string[pyarrow]
author_playtime_last_two_weeks    string[pyarrow]
author_playtime_at_review         string[pyarrow]
author_last_played                string[pyarrow]
language                          string[pyarrow]
review                            string[pyarrow]
timestamp_created                 string[pyarrow]
timestamp_updated                 string[pyarrow]
voted_up                          string[pyarrow]
votes_up                          string[pyarrow]
votes_funny                       string[pyarrow]
weighted_vote_score               string[pyarrow]
comment_co

## Consulta 2 - Eliminación de duplicados

Objetivo: eliminar registros repetidos considerando author_steamid, appid y review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [7]:
# ==========================================================
# CONSULTA 2 - ELIMINACIÓN DE DUPLICADOS CON DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")


registros_iniciales = dfd.shape[0].compute()


print("Registros iniciales:",
      registros_iniciales)


# Eliminación de duplicados

dfd_clean = dfd.drop_duplicates(
    subset=[
        "author_steamid",
        "appid",
        "review"
    ]
)


# Ejecutar cálculo

registros_finales = dfd_clean.shape[0].compute()


print("Registros después de eliminar duplicados:",
      registros_finales)


print("Duplicados eliminados:",
      registros_iniciales - registros_finales)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_2"] = fin - inicio


===== DASK =====
Registros iniciales: 500000
Registros después de eliminar duplicados: 315809
Duplicados eliminados: 184191

Tiempo de ejecución: 19.771918296813965 segundos


## Consulta 3 - Tratamiento de valores nulos

Objetivo: identificar valores faltantes y limpiar registros sin información en review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [8]:
# ==========================================================
# CONSULTA 3 - TRATAMIENTO DE VALORES NULOS CON DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")


registros_iniciales = (
    dfd_clean.shape[0]
    .compute()
)


print("Registros iniciales:",
      registros_iniciales)


print("\nValores nulos antes:")

print(
    dfd_clean.isnull()
            .sum()
            .compute()
)


# Eliminación de registros sin review

dfd_null_clean = dfd_clean.dropna(
    subset=["review"]
)


registros_finales = (
    dfd_null_clean.shape[0]
    .compute()
)


print("\nRegistros después de limpiar:",
      registros_finales)


print("Registros eliminados:",
      registros_iniciales - registros_finales)


print("\nValores nulos después:")

print(
    dfd_null_clean.isnull()
                  .sum()
                  .compute()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_3"] = fin - inicio


===== DASK =====
Registros iniciales: 315809

Valores nulos antes:
recommendationid                       0
appid                                  0
game                                  22
author_steamid                         0
author_num_games_owned                 0
author_num_reviews                     0
author_playtime_forever                0
author_playtime_last_two_weeks         0
author_playtime_at_review              0
author_last_played                     0
language                               0
review                                 0
timestamp_created                      0
timestamp_updated                      0
voted_up                               0
votes_up                               0
votes_funny                            0
weighted_vote_score                    0
comment_count                          0
steam_purchase                         0
received_for_free                      0
written_during_early_access            0
hidden_in_steam_china          

## Consulta 4 - Transformación de variables

Objetivo: crear review_length como cantidad de caracteres de cada reseña.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [9]:
# ==========================================================
# CONSULTA 4 - TRANSFORMACIÓN DE VARIABLES CON DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")


dfd_transform = dfd_null_clean.copy()


# Crear variable review_length

dfd_transform["review_length"] = (
    dfd_transform["review"]
    .str.len()
)


# Ejecutar para obtener cantidad real

registros = dfd_transform.shape[0].compute()


print("Registros procesados:",
      registros)


print("\nEjemplo de transformación:")

print(
    dfd_transform[
        [
            "review",
            "review_length"
        ]
    ].head()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_4"] = fin - inicio


===== DASK =====
Registros procesados: 315809

Ejemplo de transformación:
                                               review  review_length
0   Don't get me wrong, I love Horizon games, I've...           2184
7   一旦你点赞了，牛子精灵不仅会给你的牛子加长1cm 还可以让一个你最讨厌的人的牛子缩短1cm ...            279
10  Still Addicted  after 5 years of playing it. N...             85
16  I bought Homefront: The Revolution on a steam ...           1069
23  Oyun güzel fakat dlc leri çok pahalı su27 ve f...            134

Tiempo de ejecución: 23.632856369018555 segundos


## Consulta 5 - Filtrado de reseñas recomendadas

Objetivo: seleccionar registros donde voted_up sea igual a 1.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [10]:
# ==========================================================
# CONSULTA 5 - FILTRADO DE RESEÑAS RECOMENDADAS CON DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")


registros_iniciales = (
    dfd_transform.shape[0]
    .compute()
)


print("Registros iniciales:",
      registros_iniciales)



# Filtrar reseñas recomendadas

dfd_positive = dfd_transform[
    dfd_transform["voted_up"] == "1"
]


registros_finales = (
    dfd_positive.shape[0]
    .compute()
)


print("Registros recomendados:",
      registros_finales)


print("Porcentaje de recomendaciones:",
      (registros_finales / registros_iniciales) * 100,
      "%")


print("\nEjemplo de datos:")

print(
    dfd_positive[
        [
            "game",
            "review",
            "voted_up"
        ]
    ].head()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_5"] = fin - inicio


===== DASK =====
Registros iniciales: 315809
Registros recomendados: 266640
Porcentaje de recomendaciones: 84.43077936347603 %

Ejemplo de datos:
                          game  \
10  Sid Meier's Civilization V   
23     DCS World Steam Edition   
45                 Blush Blush   
46                Titanfall® 2   
48   The Sims™ 4 Eco Lifestyle   

                                               review voted_up  
10  Still Addicted  after 5 years of playing it. N...        1  
23  Oyun güzel fakat dlc leri çok pahalı su27 ve f...        1  
45                                         i like men        1  
46  “我不能失去第二位铁驭”有亿点感人  画面做的非常棒 （充斥着《阿凡达》《饥饿游戏》和诺兰《...        1  
48  [h1] My personal favorite expansion pack for t...        1  

Tiempo de ejecución: 35.86815142631531 segundos


## Consulta 6 - Cantidad de reseñas por videojuego

Objetivo: agrupar por game y calcular la cantidad total de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [11]:
# ==========================================================
# CONSULTA 6 - CANTIDAD DE RESEÑAS POR VIDEOJUEGO - DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")


dfd_game_reviews = (
    dfd_transform
    .groupby("game")
    .size()
    .reset_index()
)


dfd_game_reviews.columns = [
    "game",
    "total_reviews"
]


# Ejecutar cálculo distribuido

dfd_game_reviews = (
    dfd_game_reviews
    .compute()
    .sort_values(
        "total_reviews",
        ascending=False
    )
)


print("Cantidad de videojuegos procesados:",
      len(dfd_game_reviews))


print("\nTop videojuegos por cantidad de reseñas:")

print(
    dfd_game_reviews.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_6"] = fin - inicio


===== DASK =====
Cantidad de videojuegos procesados: 22991

Top videojuegos por cantidad de reseñas:
                                game  total_reviews
1215                Counter-Strike 2           1923
4506             PUBG: BATTLEGROUNDS           1513
5759                  Stardew Valley           1413
6136                        Terraria           1264
2641              Grand Theft Auto V           1228
6468        The Witcher 3: Wild Hunt           1228
6570  Tom Clancy's Rainbow Six Siege           1224
6250                      The Forest           1168
7059                Wallpaper Engine           1093
5151                            Rust           1034

Tiempo de ejecución: 12.087294101715088 segundos


## Consulta 7 - Porcentaje de recomendación por videojuego

Objetivo: calcular porcentaje de reseñas positivas por juego usando voted_up.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [12]:
# ==========================================================
# CONSULTA 7 - PORCENTAJE DE RECOMENDACIÓN POR VIDEOJUEGO
# DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")

# 1. Transformar explícitamente la columna a entero antes de agrupar
dfd_transform["voted_up"] = dfd_transform["voted_up"].astype(int)

dfd_recommendation = (
    dfd_transform
    .groupby("game")
    .agg(
        {
            "voted_up": ["count", "sum"]
        }
    )
)

dfd_recommendation.columns = [
    "total_reviews",
    "positive_reviews"
]

dfd_recommendation = (
    dfd_recommendation
    .reset_index()
)

# Ejecución distribuida
dfd_recommendation = (
    dfd_recommendation
    .compute()
)

dfd_recommendation["recommendation_percentage"] = (
    dfd_recommendation["positive_reviews"] /
    dfd_recommendation["total_reviews"]
    * 100
)

dfd_recommendation = (
    dfd_recommendation
    .sort_values(
        "recommendation_percentage",
        ascending=False
    )
)

print("Videojuegos procesados:",
      len(dfd_recommendation))

print("\nTop juegos recomendados:")
print(
    dfd_recommendation.head(10)
)

fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_7"] = fin - inicio


===== DASK =====
Videojuegos procesados: 22991

Top juegos recomendados:
                                         game  total_reviews  \
22990                              Last Dream              1   
22989                               Ai no Uta              1   
22988                   Suits: Absolute Power              1   
22987               The Sekimeiya: Spun Glass              1   
22986        The Legend of Bear-Truck Trucker              1   
22985  PAYDAY 2: The Golden Grin Casino Heist              1   
22984             Rome: Total War - Alexander              1   
22943                       The Spiral Scouts              1   
22945                       Alpha Hole Prison              1   
22946                  DCS: P-47D Thunderbolt              1   

       positive_reviews  recommendation_percentage  
22990                 1                      100.0  
22989                 1                      100.0  
22988                 1                      100.0  
22987     

## Consulta 8 - Promedio de horas jugadas por videojuego

Objetivo: calcular promedio de author_playtime_forever por juego.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [13]:
# ==========================================================
# CONSULTA 8 - PROMEDIO DE HORAS JUGADAS POR VIDEOJUEGO
# DASK
# ==========================================================

import time

inicio = time.time()

print("===== DASK =====")

# 1. Transformar la columna a formato numérico (float)
dfd_transform["author_playtime_forever"] = dfd_transform["author_playtime_forever"].astype(float)

dfd_playtime = (
    dfd_transform
    .groupby("game")
    ["author_playtime_forever"]
    .mean()
    .reset_index()
)

dfd_playtime.columns = [
    "game",
    "avg_playtime_minutes"
]

# Ejecutar cálculo distribuido
dfd_playtime = (
    dfd_playtime
    .compute()
    .sort_values(
        "avg_playtime_minutes",
        ascending=False
    )
)

print("Videojuegos procesados:",
      len(dfd_playtime))

print("\nVideojuegos con mayor promedio de juego:")
print(
    dfd_playtime.head(10)
)

fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_8"] = fin - inicio


===== DASK =====
Videojuegos procesados: 22992

Videojuegos con mayor promedio de juego:
                                     game  avg_playtime_minutes
21617       Oops!!! I Slept With Your Mom             2181697.0
19239                 Legions of Ashworld             1982834.0
337                    Aquarium Simulator             1687347.0
18466                            WalkinVR             1366674.0
2973                        Houdini Indie             1264521.5
22581  Oh, you touch my balls ( ͡° ͜ʖ ͡°)             1247696.0
18359     The Putinland: Divide & Conquer             1114523.0
16280                        MachineCraft              987286.0
5625                 Solitaire Forever II              945538.0
15776         Crusaders of the Lost Idols              903523.0

Tiempo de ejecución: 12.250583171844482 segundos


## Consulta 9 - Longitud promedio de reseñas por videojuego

Objetivo: calcular promedio de review_length agrupado por game.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [14]:
# ==========================================================
# CONSULTA 9 - LONGITUD PROMEDIO DE RESEÑAS POR VIDEOJUEGO
# DASK
# ==========================================================

import time


inicio = time.time()

print("===== DASK =====")


dfd_review_length = (
    dfd_transform
    .groupby("game")
    ["review_length"]
    .mean()
    .reset_index()
)


dfd_review_length.columns = [
    "game",
    "avg_review_length"
]


# Ejecutar cálculo distribuido

dfd_review_length = (
    dfd_review_length
    .compute()
    .sort_values(
        "avg_review_length",
        ascending=False
    )
)


print("Videojuegos procesados:",
      len(dfd_review_length))


print("\nVideojuegos con reseñas más extensas:")

print(
    dfd_review_length.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_9"] = fin - inicio


===== DASK =====
Videojuegos procesados: 22992

Videojuegos con reseñas más extensas:
                         game  avg_review_length
9899   Qbeh-1: The Atlas Cube             8000.0
16293          Mary Skelter 2             8000.0
22816        The Knight Witch             8000.0
20417       Indie Game Battle             8000.0
15359          Wayward Strand             8000.0
18897            Deadly Flare             7999.0
17279            Azusa Online             7999.0
7903           Alterium Shift             7998.0
15112           The Companion             7996.0
15892                Dynopunk             7996.0

Tiempo de ejecución: 12.346447706222534 segundos


## Consulta 10 - Ranking de videojuegos

Objetivo: ordenar videojuegos considerando porcentaje de recomendación y cantidad de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [15]:
# ==========================================================
# CONSULTA 10 - RANKING DE VIDEOJUEGOS
# DASK
# ==========================================================

import time


inicio = time.time()

print("===== DASK =====")


dfd_ranking = (
    dfd_transform
    .groupby("game")
    .agg(
        {
            "voted_up": ["count", "sum"]
        }
    )
)


dfd_ranking.columns = [
    "total_reviews",
    "positive_reviews"
]


dfd_ranking = (
    dfd_ranking
    .reset_index()
    .compute()
)


dfd_ranking["recommendation_percentage"] = (
    dfd_ranking["positive_reviews"] /
    dfd_ranking["total_reviews"]
    * 100
)


dfd_ranking = (
    dfd_ranking[
        dfd_ranking["total_reviews"] >= 100
    ]
    .sort_values(
        [
            "recommendation_percentage",
            "total_reviews"
        ],
        ascending=False
    )
)


print("Videojuegos rankeados:",
      len(dfd_ranking))


print("\nTop videojuegos:")

print(
    dfd_ranking.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Dask_Consulta_10"] = fin - inicio


===== DASK =====
Videojuegos rankeados: 609

Top videojuegos:
                                      game  total_reviews  positive_reviews  \
283                          Left 4 Dead 2            821               821   
704                             Subnautica            807               807   
94                                  Mirror            604               604   
372   Plants vs. Zombies: Game of the Year            532               532   
62                  Mount & Blade: Warband            428               428   
1167                                 Hades            390               390   
540                         Counter-Strike            386               386   
1329                          Satisfactory            384               384   
184                                   DOOM            382               382   
1730                        Papers, Please            356               356   

      recommendation_percentage  
283                       100.0  


In [16]:
# ==========================================================
# EXPORTACION AUTOMATIZADA DE RESULTADOS A GOOGLE CLOUD STORAGE
# ==========================================================

!pip install -q gcsfs fsspec

import pandas as pd

framework = "dask"

df_tiempos = pd.DataFrame(
    list(tiempos_resultados.items()),
    columns=["Consulta", "Tiempo_segundos"]
)

print(df_tiempos)

ruta_salida = f"gs://bigdata-2026-02/proyecto01/tiempos_{framework}_{arquitectura}.csv"

df_tiempos.to_csv(ruta_salida, index=False)

print(f"\nResultados exportados a: {ruta_salida}")


           Consulta  Tiempo_segundos
0   Dask_Consulta_1        16.391944
1   Dask_Consulta_2        19.771918
2   Dask_Consulta_3        50.682993
3   Dask_Consulta_4        23.632856
4   Dask_Consulta_5        35.868151
5   Dask_Consulta_6        12.087294
6   Dask_Consulta_7        12.878385
7   Dask_Consulta_8        12.250583
8   Dask_Consulta_9        12.346448
9  Dask_Consulta_10        12.576328

Resultados exportados a: gs://bigdata-2026-02/proyecto01/tiempos_dask_4_workers.csv
